**BAGGING**

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

In [ ]:
data = load_breast_cancer()
X,y = data.data,data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
single = DecisionTreeClassifier(random_state=42).fit(X_train,y_train)

print("Single tree :", round(accuracy_score(y_test, single.predict(X_test)), 4))

Single tree : 0.9123


In [4]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

bag.fit(X_train,y_train)
print("Bagging (100):", round(accuracy_score(y_test, bag.predict(X_test)), 4))
print("OOB score   :", round(bag.oob_score_, 4))


Bagging (100): 0.9386
OOB score   : 0.956


**RANDOM FOREST**


In [6]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [7]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)

In [ ]:
#For comparison
bag = BaggingClassifier(DecisionTreeClassifier(random_state=42),
                        n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
print("Bagging      :", round(accuracy_score(y_test, bag.predict(X_test)), 4))

Bagging      : 0.9386


In [11]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest:", round(accuracy_score(y_test, rf.predict(X_test)), 4))
print("OOB score is : ",round(rf.oob_score_,4))

Random Forest: 0.9561
OOB score is :  0.9538


**Gradient Boosting**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [14]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train,y_train)
print("Gradient Boosting:", round(accuracy_score(y_test, gb.predict(X_test)), 4))

Gradient Boosting: 0.9561


**ALL FOUR COMPARISON**

In [16]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              GradientBoostingClassifier)
from sklearn.metrics import accuracy_score
import numpy as np

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)

# Define all four models
models = {
    "Single Tree (tuned)":  DecisionTreeClassifier(max_depth=3, random_state=42),
    "Bagging":              BaggingClassifier(DecisionTreeClassifier(random_state=42),
                                              n_estimators=100, random_state=42, n_jobs=-1),
    "Random Forest":        RandomForestClassifier(n_estimators=100,max_features='sqrt', random_state=42, n_jobs=-1),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                       max_depth=3, random_state=42),
}

print(f"{'Model':<22}{'Test Acc':>10}{'CV Mean':>10}{'CV Std':>9}")
print("-" * 51)
for name, model in models.items():
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    # 5-fold cross-validation on TRAIN — more reliable than a single test number
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name:<22}{test_acc:>10.4f}{cv.mean():>10.4f}{cv.std():>9.4f}")

Model                   Test Acc   CV Mean   CV Std
---------------------------------------------------
Single Tree (tuned)       0.9386    0.9253   0.0245
Bagging                   0.9386    0.9516   0.0204
Random Forest             0.9561    0.9538   0.0235
Gradient Boosting         0.9561    0.9560   0.0139
